# NBA Salary Prediction - Data Understanding

Goal: Understand the structure and quality of both datasets before 
merging and modeling.

Data sources:
- Stats: Basketball Reference per 100 possessions for the 2024-25 season
- Salaries: Basketball Reference contracts for the 2025-26 season

Modeling logic: Teams sign players based on past performance. Using 
2024-25 stats to predict 2025-26 salary mirrors how real contracts work.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
DATA_DIR = Path('../data')

player_season_stats = pd.read_csv(DATA_DIR / '24-25_season_stats_per_100_possesions.csv')
player_salary = pd.read_csv(DATA_DIR / '25-26_nba_salary_contracts.csv')

print(f"Player Season Stats: {player_season_stats.shape[0]} rows, {player_season_stats.shape[1]} columns")
print(f"Player Salary: {player_salary.shape[0]} rows, {player_salary.shape[1]} columns")

Player Season Stats: 735 rows, 34 columns
Player Salary: 524 rows, 11 columns


## Player Season Stats - First Look

In [5]:
player_season_stats.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,AST,STL,BLK,TOV,PF,PTS,ORtg,DRtg,Awards,Player-ID
0,1,Mikal Bridges,28,NYK,SF,82,82,3036,9.7,19.3,...,5.0,1.2,0.7,2.2,2.1,23.6,117,118,NaN,bridgmi01
1,2,Josh Hart,29,NYK,SG,77,77,2897,6.9,13.2,...,7.8,2.0,0.5,2.7,3.4,18.0,125,112,NaN,hartjo01
2,3,Anthony Edwards,23,MIN,SG,79,79,2871,12.4,27.7,...,6.2,1.6,0.9,4.3,2.6,37.4,115,112,MVP-7CPOY-3ASNBA2,edwaran01
3,4,Devin Booker,28,PHO,SG,75,75,2795,11.6,25.1,...,9.4,1.2,0.3,3.9,3.5,34.0,119,123,NaN,bookede01
4,5,James Harden,35,LAC,PG,79,79,2789,9.4,22.9,...,12.1,2.1,1.0,6.0,2.9,31.8,114,110,MVP-10ASNBA3,hardeja01


## Player Salary - First Look

In [6]:
player_salary.head()

,Rk,Player,Tm,2025-26,2026-27,2027-28,2028-29,2029-30,2030-31,Guaranteed,Player_ID
0,1,Stephen Curry,GSW,$59606817,$62587158,NaN,NaN,NaN,NaN,$122193975,curryst01
1,2,Joel Embiid,PHI,$55224526,$58100000,$62748000,$67396000,NaN,NaN,$176072526,embiijo01
2,3,Nikola Jokić,DEN,$55224526,$59033114,$62841702,NaN,NaN,NaN,$114257640,jokicni01
3,4,Kevin Durant,HOU,$54708609,$43902439,$46097561,NaN,NaN,NaN,$98611048,duranke01
4,5,Jayson Tatum,BOS,$54126450,$58456566,$62786682,$67116798,$71446914,NaN,$242486496,tatumja01


## Column names and data types

In [9]:
print("----- Player Season Stats Columns and Data Types ------")
print(player_season_stats.dtypes)

----- Player Season Stats Columns and Data Types ------
Rk             int64
Player           str
Age            int64
Team             str
Pos              str
G              int64
GS             int64
MP             int64
FG           float64
FGA          float64
FG%          float64
3P           float64
3PA          float64
3P%          float64
2P           float64
2PA          float64
2P%          float64
eFG%         float64
FT           float64
FTA          float64
FT%          float64
ORB          float64
DRB          float64
TRB          float64
AST          float64
STL          float64
BLK          float64
TOV          float64
PF           float64
PTS          float64
ORtg           int64
DRtg           int64
Awards           str
Player-ID        str
dtype: object


In [8]:
print("----- Player Salary Columns and Data Types ------")
print(player_salary.dtypes)

----- Player Salary Columns and Data Types ------
Rk            int64
Player          str
Tm              str
2025-26         str
2026-27         str
2027-28         str
2028-29         str
2029-30         str
2030-31         str
Guaranteed      str
Player_ID       str
dtype: object


## Missing Values

In [14]:
season_stat_nulls = player_season_stats.isnull().sum()
print("----- Player Season Stats Nulls ------")
print(season_stat_nulls[season_stat_nulls > 0])

----- Player Season Stats Nulls ------
FG%         4
3P%        45
2P%        11
eFG%        4
FT%        42
Awards    682
dtype: int64


In [11]:
salary_nulls = player_salary.isnull().sum()
print("----- Player Salary Nulls ------")
print(salary_nulls[salary_nulls > 0])

----- Player Salary Nulls ------
2025-26         1
2026-27       174
2027-28       305
2028-29       413
2029-30       492
2030-31       517
Guaranteed     19
dtype: int64


## Traded players (duplicate check)
Players traded mid-season appear multiple times in the stats dataset.
Basketball Reference uses 2TM or 3TM to represent combined season totals.
We will keep only the combined row for each traded player.

In [12]:
# Players appearing more than once
traded = player_season_stats[player_season_stats.duplicated('Player', keep=False)]
print(f"Rows involving traded players: {len(traded)}")
print(f"\nUnique traded players: {traded['Player'].nunique()}")
print(f"\nExample — how traded player rows look:")
print(traded[['Player', 'Team', 'G', 'PTS']].head(9).to_string())

Rows involving traded players: 247

Unique traded players: 81

Example — how traded player rows look:
          Player Team   G   PTS
16   Zach LaVine  2TM  74  31.6
17   Zach LaVine  CHI  42  32.8
18   Zach LaVine  SAC  32  30.0
53  De'Aaron Fox  2TM  62  31.7
54  De'Aaron Fox  SAC  45  33.1
55  De'Aaron Fox  SAS  17  28.0
75  Max Christie  2TM  78  17.2
76  Max Christie  LAL  46  16.6
77  Max Christie  DAL  32  17.8


## Salary target variable (cleaning needed)

In [13]:
print("Sample salary values (raw):")
print(player_salary[['Player', 'Tm', '2025-26']].head(10).to_string())
print(f"\nDtype: {player_salary['2025-26'].dtype}")

Sample salary values (raw):
                  Player   Tm    2025-26
0          Stephen Curry  GSW  $59606817
1            Joel Embiid  PHI  $55224526
2           Nikola Jokić  DEN  $55224526
3           Kevin Durant  HOU  $54708609
4           Jayson Tatum  BOS  $54126450
5          Anthony Davis  WAS  $54126450
6  Giannis Antetokounmpo  MIL  $54126450
7           Jimmy Butler  GSW  $54126450
8           Jaylen Brown  BOS  $53142264
9           Devin Booker  PHO  $53142264

Dtype: str


## Data understanding findings

### Stats dataset
- 735 rows but only 488 unique players — 81 traded players appear 
  multiple times (247 extra rows). We keep only the 2TM/3TM combined row.
- Shooting % nulls (FG%, 3P%, 2P%, FT%): players who had zero attempts 
  in that category — not bad data, expected. Will fill with 0.
- Awards: 682 nulls — most players won no awards. Will drop this column,
  it is not a performance stat and would cause data leakage.

### Salary dataset
- We only need the 2025-26 column — all future year nulls are irrelevant
- 1 null in 2025-26 — will drop that row
- Salary stored as string with $ signs — needs converting to numeric
- We will drop all future year columns and Guaranteed column

### Merge strategy
- Merge on Player-ID (stats) = Player_ID (salary) — exact ID match,
  avoids name spelling mismatches
- Inner join — only keep players present in both datasets
- Expected result: ~400-450 players after removing traded duplicates 
  and players without 2025-26 contracts

In [15]:
# Confirm merge key exists in both
print("Stats merge key sample:")
print(player_season_stats['Player-ID'].head(5).tolist())

print("\nSalary merge key sample:")
print(player_salary['Player_ID'].head(5).tolist())

# Check for any players in salary missing from stats
salary_ids  = set(player_salary['Player_ID'].dropna())
stats_ids   = set(player_season_stats['Player-ID'].dropna())

print(f"\nPlayers in salary but not in stats: {len(salary_ids - stats_ids)}")
print(f"Players in stats but not in salary: {len(stats_ids - salary_ids)}")
print(f"Players in both: {len(salary_ids & stats_ids)}")

Stats merge key sample:
['bridgmi01', 'hartjo01', 'edwaran01', 'bookede01', 'hardeja01']

Salary merge key sample:
['curryst01', 'embiijo01', 'jokicni01', 'duranke01', 'tatumja01']

Players in salary but not in stats: 64
Players in stats but not in salary: 145
Players in both: 424


## Merge analysis

424 players appear in both datasets — our final modeling dataset.

Players in salary but not stats (64): players who have 2025-26 contracts 
but did not play in 2024-25 — injured players like Kawhi Leonard. 
We cannot include them as we have no performance data to model from.

Players in stats but not salary (145): players who played in 2024-25 
but have no 2025-26 contract yet — free agents, two-way contracts, 
or players who were waived. Correctly excluded from the model.

424 players with both performance stats and confirmed contracts is a 
strong and realistic modeling dataset.

## Next steps

- Clean salary to numeric ($ removal) for visualization purposes only
- Filter to 424 players present in both datasets
- Handle traded players — keep 2TM/3TM combined row
- Visualize salary distribution
- Explore salary by position
- Explore which stats correlate strongest with salary
- Investigate overpaid vs underpaid players
- Identify key features for modeling